# Лабораторна 1 · Notebook 2 · Зовнішня перевірка запущеного API

**Питання частини:** чи працює реалізований сервіс через той самий інтерфейс, яким
користуватиметься зовнішній клієнт?

Цей notebook — невеликий HTTP-клієнт. Він **не імпортує** `app`, обробники, правила перевірки
чи сховище, не використовує `TestClient` і не звертається до провайдера LLM напряму. Ключ
провайдера потрібен лише сервісу в режимі `live`.

## Умови запуску

Перед виконанням підніміть три екземпляри сервісу в окремих терміналах (команди — у `README.md`):

| Адреса | `FAULTLOG_AI_MODE` | Сховище | Для яких сценаріїв |
|---|---|---|---|
| `http://127.0.0.1:8000` | `scripted` | `data/run-scripted.json` | S1–S8 (детерміновані відповіді моделі) |
| `http://127.0.0.1:8001` | `unavailable` | `data/run-unavailable.json` | S9–S10 (контрольований збій моделі) |
| `http://127.0.0.1:8002` | `live` | `data/run-live.json` | S11 (жива інтеграція, необов'язково) |

Кожен прохід використовує **окреме тестове сховище без цінних даних**; перевірки рахують приріст
від зафіксованого початкового стану, тому повторний запуск не потребує ендпоінта видалення.

Очікування взяті зі `specs/spec.md` (розділи 2, 4, 6), а не обчислені кодом, який перевіряється.

In [1]:
import json
from datetime import datetime, timezone

import requests
from IPython.display import Markdown, display

BASE_SCRIPTED = "http://127.0.0.1:8000"
BASE_UNAVAILABLE = "http://127.0.0.1:8001"
BASE_LIVE = "http://127.0.0.1:8002"

# Кінцевий час очікування > граничного часу обробки AI-запиту в сервісі (FAULTLOG_AI_TIMEOUT=30).
TIMEOUT = 35

print("API під перевіркою:")
print(" scripted   ", BASE_SCRIPTED)
print(" unavailable", BASE_UNAVAILABLE)
print(" live       ", BASE_LIVE)
print("timeout, с:", TIMEOUT)
print("час запуску:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

API під перевіркою:
 scripted    http://127.0.0.1:8000
 unavailable http://127.0.0.1:8001
 live        http://127.0.0.1:8002
timeout, с: 35
час запуску: 2026-10-09T14:11:40+00:00


In [2]:
RESULTS = []


def call(method, base, path, payload=None):
    """Справжній HTTP-запит. Недоступний сервер або таймаут не підміняються даними клієнта."""
    url = f"{base}{path}"
    response = requests.request(method, url, json=payload, timeout=TIMEOUT)
    try:
        body = response.json()
    except ValueError as error:
        raise AssertionError(f"{url}: відповідь не є JSON ({error})") from error
    return response.status_code, body


def fetch_list(base):
    status, body = call("GET", base, "/faults")
    assert status == 200, f"GET /faults дав {status}"
    assert set(body) == {"count", "items"}, f"несподівана структура списку: {sorted(body)}"
    assert body["count"] == len(body["items"]), "count не відповідає кількості items"
    return body["count"], body["items"]


def ids_and_fields(items):
    """Повний відпечаток списку: id разом з усіма полями (рішення C-9 у specs/clarify.md)."""
    return {item["id"]: {k: v for k, v in item.items()} for item in items}


def scenario(sid, name, criterion, ai_source):
    """Декоратор: виконує сценарій, зберігає passed/failed/not_run і не зупиняє інші сценарії."""
    def wrapper(func):
        entry = {"id": sid, "scenario": name, "criterion": criterion, "ai_source": ai_source,
                 "expected": "", "actual": "", "records": "", "result": "failed", "note": ""}
        try:
            detail = func() or {}
            entry.update(detail)
            entry["result"] = detail.get("result", "passed")
        except requests.RequestException as error:
            entry["result"] = "failed"
            entry["note"] = f"сервер недоступний або таймаут: {type(error).__name__}"
        except AssertionError as error:
            entry["result"] = "failed"
            entry["note"] = str(error)[:300]
        RESULTS.append(entry)
        print(f"[{entry['result'].upper():8}] {sid} {name} — {entry.get('note') or entry.get('actual')}")
        return func
    return wrapper

## Випадки та очікування зі специфікації

Синтетичні повідомлення збігаються з ключами фікстури `app/fixtures/scripted_proposals.json`,
тому для режиму `scripted` джерело пропозиції відоме заздалегідь.

In [3]:
MANUAL_OK = {"title": "Ручний запис: не крутить стартер", "system": "starter", "mileage_km": 99000}
MANUAL_BAD = {"title": "ок", "system": "starter"}  # title із 2 символів порушує правило поля

TEXT_VALID = "Стартер лише клацає, світло на панелі яскраве, пробіг 120000 км"
TEXT_INCOMPLETE = "Машина не їде, зробіть щось"
TEXT_DANGER = "Після спроби завести з-під капота йде дим і сильно пахне бензином"
TEXT_INVALID_PROPOSAL = "Акумулятор сів, на панелі нічого не світиться"   # фікстура: system=engine_block
TEXT_CONTENT_REJECT = "Стартер клацає, світло на панелі яскраве"          # фікстура: вигаданий пробіг 150000
TEXT_LIVE = "Після мийки двигуна мокрі свічки, мотор крутить і не заводиться"

EXPECTATIONS = {
    "S1": "200, {count, items}",
    "S2": "201 created; id у списку; поля збігаються; count +1",
    "S3": "422 validation_error; список незмінний",
    "S4": "201 created; system=starter, mileage_km=120000; рівно +1 запис",
    "S5": "422 needs_clarification + question; записи незмінні",
    "S6": "422 safety_hold; записи незмінні",
    "S7": "422 invalid_proposal (system_not_allowed:engine_block); записи незмінні",
    "S8": "422 content_rejected (mileage_not_in_text:150000); записи незмінні",
    "S9": "503 ai_unavailable; записи незмінні",
    "S10": "201 created вручну за недоступної моделі; count +1",
    "S11": "201 created через живу модель або чесний not_run",
}
EXPECTATIONS

{'S1': '200, {count, items}',
 'S2': '201 created; id у списку; поля збігаються; count +1',
 'S3': '422 validation_error; список незмінний',
 'S4': '201 created; system=starter, mileage_km=120000; рівно +1 запис',
 'S5': '422 needs_clarification + question; записи незмінні',
 'S6': '422 safety_hold; записи незмінні',
 'S7': '422 invalid_proposal (system_not_allowed:engine_block); записи незмінні',
 'S8': '422 content_rejected (mileage_not_in_text:150000); записи незмінні',
 'S9': '503 ai_unavailable; записи незмінні',
 'S10': '201 created вручну за недоступної моделі; count +1',
 'S11': '201 created через живу модель або чесний not_run'}

## S1. Доступність і початковий стан

In [4]:
@scenario("S1", "список і початковий стан", "AC-1", "n/a")
def s1():
    status, health = call("GET", BASE_SCRIPTED, "/health")
    assert status == 200, f"GET /health дав {status}"
    assert health["ai_mode"] == "scripted", f"екземпляр 8000 має бути scripted, а не {health['ai_mode']}"

    count, items = fetch_list(BASE_SCRIPTED)
    globals()["INITIAL_COUNT"] = count
    globals()["INITIAL_STATE"] = ids_and_fields(items)
    return {"expected": EXPECTATIONS["S1"], "actual": f"200, count={count}, ai_mode=scripted",
            "records": f"початковий стан зафіксовано: {count}"}

[PASSED  ] S1 список і початковий стан — 200, count=0, ai_mode=scripted


## S2. Ручне створення

In [5]:
@scenario("S2", "ручне створення", "AC-1", "n/a")
def s2():
    before_count, before_items = fetch_list(BASE_SCRIPTED)
    status, body = call("POST", BASE_SCRIPTED, "/faults", MANUAL_OK)
    assert status == 201, f"очікували 201, отримали {status}: {body}"
    assert body["status"] == "created", body
    record = body["record"]
    assert record["source"] == "manual" and record["safety_stop"] is False, record
    for field, value in MANUAL_OK.items():
        assert record[field] == value, f"{field}: {record[field]} != {value}"

    after_count, after_items = fetch_list(BASE_SCRIPTED)
    assert after_count == before_count + 1, f"count {before_count} -> {after_count}"
    stored = [item for item in after_items if item["id"] == record["id"]]
    assert stored == [record], "запис у списку має збігатися з відповіддю за всіма полями"
    globals()["MANUAL_RECORD_ID"] = record["id"]
    return {"expected": EXPECTATIONS["S2"], "actual": f"201, id={record['id'][:8]}…",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S2 ручне створення — 201, id=6dd6bbaf…


## S3. Хибний ручний вхід

In [6]:
@scenario("S3", "хибний ручний вхід", "AC-1", "n/a")
def s3():
    before_count, before_items = fetch_list(BASE_SCRIPTED)
    status, body = call("POST", BASE_SCRIPTED, "/faults", MANUAL_BAD)
    assert status == 422, f"очікували 422, отримали {status}: {body}"
    assert body["status"] == "validation_error", body
    assert body["reason"] == "title_length_or_type", body

    after_count, after_items = fetch_list(BASE_SCRIPTED)
    assert after_count == before_count, f"count змінився: {before_count} -> {after_count}"
    assert ids_and_fields(after_items) == ids_and_fields(before_items), "вміст записів змінився"
    return {"expected": EXPECTATIONS["S3"], "actual": f"422 {body['reason']}",
            "records": f"{before_count} -> {after_count} (id і поля незмінні)"}

[PASSED  ] S3 хибний ручний вхід — 422 title_length_or_type


## S4. Коректна AI-пропозиція

In [7]:
@scenario("S4", "коректна AI-пропозиція", "AC-2", "scripted")
def s4():
    before_count, before_items = fetch_list(BASE_SCRIPTED)
    status, body = call("POST", BASE_SCRIPTED, "/faults/from-text", {"text": TEXT_VALID})
    assert status == 201, f"очікували 201, отримали {status}: {body}"
    assert body["status"] == "created", body
    record = body["record"]
    assert record["system"] == "starter", record
    assert record["mileage_km"] == 120000, record
    assert record["source"] == "ai" and record["safety_stop"] is False, record

    after_count, after_items = fetch_list(BASE_SCRIPTED)
    assert after_count == before_count + 1, "має з'явитися рівно один новий запис"
    new_ids = set(ids_and_fields(after_items)) - set(ids_and_fields(before_items))
    assert new_ids == {record["id"]}, f"нові id: {new_ids}"
    assert [i for i in after_items if i["id"] == record["id"]] == [record]
    return {"expected": EXPECTATIONS["S4"], "actual": f"201, starter/120000, id={record['id'][:8]}…",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S4 коректна AI-пропозиція — 201, starter/120000, id=ba6d3496…


## S5. Неповний або неоднозначний текст

In [8]:
@scenario("S5", "неповний текст", "AC-3", "scripted")
def s5():
    before_count, before_items = fetch_list(BASE_SCRIPTED)
    status, body = call("POST", BASE_SCRIPTED, "/faults/from-text", {"text": TEXT_INCOMPLETE})
    assert status == 422, f"очікували 422, отримали {status}: {body}"
    assert body["status"] == "needs_clarification", body
    assert body.get("question"), "відповідь має містити питання для уточнення"
    assert "record" not in body, "запис не повинен створюватися"

    after_count, after_items = fetch_list(BASE_SCRIPTED)
    assert after_count == before_count and ids_and_fields(after_items) == ids_and_fields(before_items)
    return {"expected": EXPECTATIONS["S5"], "actual": f"422 needs_clarification: {body['question'][:40]}…",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S5 неповний текст — 422 needs_clarification: Що саме відбувається при повороті ключа?…


## S6. Ознака небезпеки

In [9]:
@scenario("S6", "ознака небезпеки", "AC-3", "n/a")
def s6():
    before_count, before_items = fetch_list(BASE_SCRIPTED)
    status, body = call("POST", BASE_SCRIPTED, "/faults/from-text", {"text": TEXT_DANGER})
    assert status == 422, f"очікували 422, отримали {status}: {body}"
    assert body["status"] == "safety_hold", body
    assert body["reason"] == "danger_markers_in_text", body
    assert body.get("message"), "водій має отримати повідомлення про небезпеку"

    after_count, after_items = fetch_list(BASE_SCRIPTED)
    assert after_count == before_count and ids_and_fields(after_items) == ids_and_fields(before_items)
    return {"expected": EXPECTATIONS["S6"], "actual": f"422 safety_hold ({body['reason']})",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S6 ознака небезпеки — 422 safety_hold (danger_markers_in_text)


## S7. Структурно хибна пропозиція

In [10]:
@scenario("S7", "структурно хибна пропозиція", "AC-4", "scripted")
def s7():
    before_count, before_items = fetch_list(BASE_SCRIPTED)
    status, body = call("POST", BASE_SCRIPTED, "/faults/from-text", {"text": TEXT_INVALID_PROPOSAL})
    assert status == 422, f"очікували 422, отримали {status}: {body}"
    assert body["status"] == "invalid_proposal", body
    assert body["reason"] == "system_not_allowed:engine_block", body

    after_count, after_items = fetch_list(BASE_SCRIPTED)
    assert after_count == before_count and ids_and_fields(after_items) == ids_and_fields(before_items)
    return {"expected": EXPECTATIONS["S7"], "actual": f"422 {body['reason']}",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S7 структурно хибна пропозиція — 422 system_not_allowed:engine_block

## S8. Змістовно хибна пропозиція

Пропозиція структурно правильна: `system` зі списку, `mileage_km` — ціле в межах. Відхилення
відбувається саме через зміст: числа `150000` немає в повідомленні. Порівняйте з S4, де єдина
відмінність — пробіг, підтверджений текстом.

In [11]:
@scenario("S8", "змістовно хибна пропозиція", "AC-4", "scripted")
def s8():
    before_count, before_items = fetch_list(BASE_SCRIPTED)
    status, body = call("POST", BASE_SCRIPTED, "/faults/from-text", {"text": TEXT_CONTENT_REJECT})
    assert status == 422, f"очікували 422, отримали {status}: {body}"
    assert body["status"] == "content_rejected", body
    assert body["reason"] == "mileage_not_in_text:150000", body

    after_count, after_items = fetch_list(BASE_SCRIPTED)
    assert after_count == before_count and ids_and_fields(after_items) == ids_and_fields(before_items)
    return {"expected": EXPECTATIONS["S8"], "actual": f"422 {body['reason']}",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S8 змістовно хибна пропозиція — 422 mileage_not_in_text:150000


## S9. Контрольований збій моделі

Окремий екземпляр сервісу на порту `8001` запущено з `FAULTLOG_AI_MODE=unavailable`. Підміна
працює **всередині сервісу**; notebook надсилає такий самий справжній HTTP-запит.

In [12]:
@scenario("S9", "збій моделі", "AC-5", "imitated")
def s9():
    status, health = call("GET", BASE_UNAVAILABLE, "/health")
    assert status == 200 and health["ai_mode"] == "unavailable", health

    before_count, before_items = fetch_list(BASE_UNAVAILABLE)
    status, body = call("POST", BASE_UNAVAILABLE, "/faults/from-text", {"text": TEXT_VALID})
    assert status == 503, f"очікували 503, отримали {status}: {body}"
    assert body["status"] == "ai_unavailable", body

    after_count, after_items = fetch_list(BASE_UNAVAILABLE)
    assert after_count == before_count and ids_and_fields(after_items) == ids_and_fields(before_items)
    return {"expected": EXPECTATIONS["S9"], "actual": f"503 {body['status']} ({body['reason']})",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S9 збій моделі — 503 ai_unavailable (ai_mode_unavailable)


## S10. Ручний шлях після збою моделі

In [13]:
@scenario("S10", "ручний шлях після збою", "AC-5", "imitated + n/a")
def s10():
    before_count, before_items = fetch_list(BASE_UNAVAILABLE)
    payload = {"title": "Ручний запис за недоступної моделі", "system": "battery", "mileage_km": None}
    status, body = call("POST", BASE_UNAVAILABLE, "/faults", payload)
    assert status == 201, f"очікували 201, отримали {status}: {body}"
    record = body["record"]

    after_count, after_items = fetch_list(BASE_UNAVAILABLE)
    assert after_count == before_count + 1, f"count {before_count} -> {after_count}"
    assert [i for i in after_items if i["id"] == record["id"]] == [record]
    return {"expected": EXPECTATIONS["S10"], "actual": f"201, id={record['id'][:8]}…",
            "records": f"{before_count} -> {after_count}"}

[PASSED  ] S10 ручний шлях після збою — 201, id=10a961f6…


## S11. Жива інтеграція через сервіс

Окремий екземпляр на порту `8002` працює з `FAULTLOG_AI_MODE=live`. Це єдиний сценарій із
реальною моделью. Якщо екземпляр не запущено або провайдер недоступний, результат — `not_run`
із поясненням; підготовленою відповіддю він не підміняється.

Реальний виклик у першому notebook **не** є доказом цієї інтеграції.

In [14]:
@scenario("S11", "жива інтеграція", "—", "live")
def s11():
    try:
        status, health = call("GET", BASE_LIVE, "/health")
    except requests.RequestException as error:
        return {"expected": EXPECTATIONS["S11"], "result": "not_run",
                "actual": "екземпляр live не запущено",
                "note": f"не виконано: {type(error).__name__}"}

    assert status == 200 and health["ai_mode"] == "live", health
    before_count, _ = fetch_list(BASE_LIVE)
    status, body = call("POST", BASE_LIVE, "/faults/from-text", {"text": TEXT_LIVE})

    if status == 503:
        return {"expected": EXPECTATIONS["S11"], "result": "not_run",
                "actual": f"503 {body.get('reason')}",
                "note": "провайдер недоступний: доказ живої інтеграції відсутній"}

    assert status == 201, f"жива модель дала {status}: {body}"
    record = body["record"]
    assert record["system"] in ["battery", "starter", "fuel", "ignition", "immobilizer"], record
    assert record["source"] == "ai", record

    after_count, after_items = fetch_list(BASE_LIVE)
    assert after_count == before_count + 1
    assert [i for i in after_items if i["id"] == record["id"]] == [record]
    return {"expected": EXPECTATIONS["S11"],
            "actual": f"201, system={record['system']}, mileage_km={record['mileage_km']}",
            "records": f"{before_count} -> {after_count}",
            "note": "жива модель; стабільність на інших входах не доведена"}

[PASSED  ] S11 жива інтеграція — жива модель; стабільність на інших входах не доведена


## Підсумок перевірок

In [15]:
header = ("| сценарій | критерій | джерело AI | очікування | фактично | записи | результат | примітка |\n"
          "|---|---|---|---|---|---|:-:|---|")
rows = [
    f"| {r['id']} {r['scenario']} | {r['criterion']} | {r['ai_source']} | {r['expected']} "
    f"| {r['actual']} | {r['records']} | {r['result']} | {r['note']} |"
    for r in RESULTS
]
display(Markdown("\n".join([header, *rows])))

counts = {
    "passed": sum(1 for r in RESULTS if r["result"] == "passed"),
    "failed": sum(1 for r in RESULTS if r["result"] == "failed"),
    "not_run": sum(1 for r in RESULTS if r["result"] == "not_run"),
}
counts

| сценарій | критерій | джерело AI | очікування | фактично | записи | результат | примітка |
|---|---|---|---|---|---|:-:|---|
| S1 список і початковий стан | AC-1 | n/a | 200, {count, items} | 200, count=0, ai_mode=scripted | початковий стан зафіксовано: 0 | passed |  |
| S2 ручне створення | AC-1 | n/a | 201 created; id у списку; поля збігаються; count +1 | 201, id=6dd6bbaf… | 0 -> 1 | passed |  |
| S3 хибний ручний вхід | AC-1 | n/a | 422 validation_error; список незмінний | 422 title_length_or_type | 1 -> 1 (id і поля незмінні) | passed |  |
| S4 коректна AI-пропозиція | AC-2 | scripted | 201 created; system=starter, mileage_km=120000; рівно +1 запис | 201, starter/120000, id=ba6d3496… | 1 -> 2 | passed |  |
| S5 неповний текст | AC-3 | scripted | 422 needs_clarification + question; записи незмінні | 422 needs_clarification: Що саме відбувається при повороті ключа?… | 2 -> 2 | passed |  |
| S6 ознака небезпеки | AC-3 | n/a | 422 safety_hold; записи незмінні | 422 safety_hold (danger_markers_in_text) | 2 -> 2 | passed |  |
| S7 структурно хибна пропозиція | AC-4 | scripted | 422 invalid_proposal (system_not_allowed:engine_block); записи незмінні | 422 system_not_allowed:engine_block | 2 -> 2 | passed |  |
| S8 змістовно хибна пропозиція | AC-4 | scripted | 422 content_rejected (mileage_not_in_text:150000); записи незмінні | 422 mileage_not_in_text:150000 | 2 -> 2 | passed |  |
| S9 збій моделі | AC-5 | imitated | 503 ai_unavailable; записи незмінні | 503 ai_unavailable (ai_mode_unavailable) | 0 -> 0 | passed |  |
| S10 ручний шлях після збою | AC-5 | imitated + n/a | 201 created вручну за недоступної моделі; count +1 | 201, id=10a961f6… | 0 -> 1 | passed |  |
| S11 жива інтеграція | — | live | 201 created через живу модель або чесний not_run | 201, system=ignition, mileage_km=None | 0 -> 1 | passed | жива модель; стабільність на інших входах не доведена |

{'passed': 11, 'failed': 0, 'not_run': 0}

In [16]:
DETERMINISTIC = [f"S{i}" for i in range(1, 11)]
failed_deterministic = [r["id"] for r in RESULTS if r["id"] in DETERMINISTIC and r["result"] != "passed"]

print("детерміновані сценарії S1-S10:", "усі passed" if not failed_deterministic else failed_deterministic)
print("жива інтеграція S11:", [r["result"] for r in RESULTS if r["id"] == "S11"])
assert not failed_deterministic, f"обов'язкові сценарії не пройшли: {failed_deterministic}"

детерміновані сценарії S1-S10: усі passed
жива інтеграція S11: ['passed']


## Висновок

**Фактичний результат цього проходу: 11 passed, 0 failed, 0 not_run.** Усі обов'язкові
детерміновані сценарії `S1`–`S10` пройшли; `S11` виконано з реальною моделлю.

**Що саме підтвердив зовнішній клієнт.** Окремо запущений сервіс відповідає за специфікацією
через HTTP: маршрути, коди, поля відповідей і фактичні зміни у сховищі. Для кожної відмови
(`S3`, `S5`–`S9`) список до і після порівняно за `id` **і** за всіма полями: жоден запис не
додано, не змінено й не перезаписано.

**Три різні докази, які не замінюють один одного.**

1. `notebooks/01_experiment.ipynb` встановив межу правил і LLM та правила перевірки пропозиції
   (9/9 детермінованих перевірок, правила 8/10, LLM 9/10).
2. `uv run pytest -q` (21 passed) перевірив мій код і контракт на контрольованих відповідях.
3. Цей notebook перевірив **запущений** сервіс як зовнішній клієнт, без імпорту його коду.

**Що показали окремі сценарії.**

- `S4` проти `S8`: пропозиції відрізняються лише значенням пробігу. Підтверджений текстом
  `120000` зберігається, вигаданий `150000` відхиляється з `mileage_not_in_text:150000`.
  Схема в обох випадках правильна — працює саме змістова перевірка.
- `S6`: для тексту з ознакою небезпеки модель не викликається взагалі, рішення приймає програма.
- `S9` і `S10`: за недоступної моделі AI-запит дає `503` без зміни даних, а ручне створення й
  список працюють і підтверджують новий запис.

**Чого цей прохід не доводить.** `S11` — один успішний виклик на одному вході: це доказ зв'язку
сервісу з провайдером, а не стабільності моделі. Детерміновані сценарії доводять поведінку
сервісу **за заданої пропозиції**, а не якість моделі. Поза перевіркою лишаються реальні
повідомлення водіїв, вартість, затримка під навантаженням і поведінка після зміни моделі.

**Помилок, виявлених цим notebook, немає.** Дві розбіжності реалізації зі специфікацією знайшли
раніше автоматичні тести сервісу; вони виправлені й описані в `specs/converge.md` (D-1, D-2).